<a href="https://colab.research.google.com/github/mena-04/FalconFoundry/blob/main/FP_Softmax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
PICO-Softmax floating-point golden model: pure base-2 Softmax.

This model intentionally computes

    y_i = 2^(x_i - max(x)) / sum_j 2^(x_j - max(x))

No log2(e) input scaling is applied. Use this file as the mathematical
reference for the fixed-point base-2 accelerator model.
"""

from __future__ import annotations

import argparse
import csv
from pathlib import Path
from typing import Any

import numpy as np

In [ ]:
def softmax_base2(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable base-2 Softmax using floating-point arithmetic."""
    x = np.asarray(x, dtype=np.float64)
    x_shift = x - np.max(x, axis=axis, keepdims=True)
    numerator = np.exp2(x_shift)
    denominator = np.sum(numerator, axis=axis, keepdims=True)
    return numerator / denominator

In [ ]:
def softmax_natural(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Standard natural-exponential Softmax, only for comparison."""
    x = np.asarray(x, dtype=np.float64)
    x_shift = x - np.max(x, axis=axis, keepdims=True)
    numerator = np.exp(x_shift)
    denominator = np.sum(numerator, axis=axis, keepdims=True)
    return numerator / denominator

In [ ]:
def entropy(p: np.ndarray, axis: int = -1) -> np.ndarray:
    """Entropy in bits. Useful because base-2 Softmax is usually flatter."""
    p = np.asarray(p, dtype=np.float64)
    p_safe = np.clip(p, 1e-300, 1.0)
    return -np.sum(p_safe * np.log2(p_safe), axis=axis)

In [ ]:
def comparison_metrics(
    *,
    samples: int = 10_000,
    vector_len: int = 8,
    seed: int = 1,
    input_std: float = 1.0,
) -> dict[str, Any]:
    """
    Compare pure base-2 Softmax with natural Softmax on random vectors.

    This is not the fixed-point accuracy test. It only shows the algorithmic
    difference caused by choosing 2^x instead of e^x.
    """
    rng = np.random.default_rng(seed)
    x = rng.normal(loc=0.0, scale=input_std, size=(samples, vector_len))

    y_base2 = softmax_base2(x, axis=1)
    y_nat = softmax_natural(x, axis=1)

    abs_err = np.abs(y_base2 - y_nat)
    return {
        "samples": samples,
        "vector_len": vector_len,
        "input_std": input_std,
        "mean_abs_difference_vs_natural_softmax": float(np.mean(abs_err)),
        "max_abs_difference_vs_natural_softmax": float(np.max(abs_err)),
        "top1_match_rate_vs_natural_softmax": float(
            np.mean(np.argmax(y_base2, axis=1) == np.argmax(y_nat, axis=1))
        ),
        "mean_entropy_bits_base2": float(np.mean(entropy(y_base2, axis=1))),
        "mean_entropy_bits_natural": float(np.mean(entropy(y_nat, axis=1))),
    }


In [ ]:
def write_float_vectors_csv(
    path: str | Path,
    *,
    num_vectors: int = 16,
    vector_len: int = 8,
    seed: int = 2,
    input_std: float = 1.0,
) -> None:
    """Write example floating-point vectors and golden base-2 outputs."""
    path = Path(path)
    rng = np.random.default_rng(seed)
    x = rng.normal(loc=0.0, scale=input_std, size=(num_vectors, vector_len))
    y_base2 = softmax_base2(x, axis=1)
    y_nat = softmax_natural(x, axis=1)

    with path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "vector_id",
            "index",
            "x_float",
            "y_base2_float_golden",
            "y_natural_float_for_comparison",
        ])
        for vector_id in range(num_vectors):
            for index in range(vector_len):
                writer.writerow([
                    vector_id,
                    index,
                    f"{x[vector_id, index]:.12g}",
                    f"{y_base2[vector_id, index]:.12g}",
                    f"{y_nat[vector_id, index]:.12g}",
                ])

In [ ]:
def demo_vector() -> None:
    """Print one simple example vector."""
    x = np.array([1.25, -0.50, 0.75, 2.00, -1.00, 0.00, 0.50, -2.00])
    y_base2 = softmax_base2(x)
    y_nat = softmax_natural(x)

    np.set_printoptions(precision=8, suppress=True)
    print("Input x:")
    print(x)
    print("\nFloating-point pure base-2 Softmax:")
    print(y_base2)
    print("sum =", np.sum(y_base2))
    print("\nNatural Softmax, comparison only:")
    print(y_nat)
    print("sum =", np.sum(y_nat))
    print("\nDifference base2 - natural:")
    print(y_base2 - y_nat)

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(
        description="Floating-point pure base-2 Softmax golden model."
    )
    parser.add_argument("--samples", type=int, default=10_000)
    parser.add_argument("--vector-len", type=int, default=8)
    parser.add_argument("--seed", type=int, default=1)
    parser.add_argument("--input-std", type=float, default=1.0)
    parser.add_argument(
        "--csv",
        type=str,
        default="",
        help="Optional CSV path for generated floating-point test vectors.",
    )
    parser.add_argument("--csv-vectors", type=int, default=16)
    args = parser.parse_args()

    demo_vector()

    print("\nRandom comparison against natural Softmax:")
    metrics = comparison_metrics(
        samples=args.samples,
        vector_len=args.vector_len,
        seed=args.seed,
        input_std=args.input_std,
    )
    for key, value in metrics.items():
        print(f"  {key}: {value}")

    if args.csv:
        write_float_vectors_csv(
            args.csv,
            num_vectors=args.csv_vectors,
            vector_len=args.vector_len,
            seed=args.seed + 100,
            input_std=args.input_std,
        )
        print(f"\nWrote {args.csv}")

In [ ]:
import sys

# Store the original sys.argv
_original_argv = sys.argv

# Set sys.argv to mimic a call with no command-line arguments.
# This prevents argparse from seeing arguments passed by the Jupyter kernel.
sys.argv = [sys.argv[0]]

try:
    if __name__ == "__main__":
        main()
except SystemExit as e:
    # Catch SystemExit which argparse raises on error (e.g., unrecognized args).
    # This prevents the kernel from stopping if an error still occurs, allowing for debugging.
    if e.code != 0:
        print(f"An error occurred during main() execution: {e}")
finally:
    # Restore sys.argv to its original state to avoid affecting other parts of the notebook.
    sys.argv = _original_argv


Input x:
[ 1.25 -0.5   0.75  2.   -1.    0.    0.5  -2.  ]

Floating-point pure base-2 Softmax:
[0.19933862 0.05926373 0.14095369 0.33524627 0.04190578 0.08381157
 0.11852745 0.02095289]
sum = 0.9999999999999998

Natural Softmax, comparison only:
[0.20831817 0.03620027 0.12635136 0.44100957 0.02195657 0.05968415
 0.09840254 0.00807737]
sum = 1.0

Difference base2 - natural:
[-0.00897955  0.02306346  0.01460233 -0.1057633   0.01994921  0.02412741
  0.02012492  0.01287552]

Random comparison against natural Softmax:
  samples: 10000
  vector_len: 8
  input_std: 1.0
  mean_abs_difference_vs_natural_softmax: 0.02472245217526389
  max_abs_difference_vs_natural_softmax: 0.2083944934302533
  top1_match_rate_vs_natural_softmax: 1.0
  mean_entropy_bits_base2: 2.7242018355901116
  mean_entropy_bits_natural: 2.4833533399871865


In [ ]:
import numpy as np
import pandas as pd

input_scale = 16.0

test_vectors_q = {
    "Test 1 - All equal": np.array(
        [16, 16, 16, 16, 16, 16, 16, 16],
        dtype=np.int64,
    ),

    "Test 2 - Increasing": np.array(
        [16, 32, 48, 64, 80, 96, 112, 127],
        dtype=np.int64,
    ),

    "Test 3 - Decreasing": np.array(
        [127, 112, 96, 80, 64, 48, 32, 16],
        dtype=np.int64,
    ),

    "Test 4 - Negative": np.array(
        [-128, -112, -96, -80, -64, -48, -32, -16],
        dtype=np.int64,
    ),

    "Test 5 - Mixed": np.array(
        [32, -16, 112, 0, -64, 80, 16, -32],
        dtype=np.int64,
    ),

    "Test 6 - Dominant": np.array(
        [112, 0, 0, 0, 0, 0, 0, 0],
        dtype=np.int64,
    ),
}

test_vectors_real = {
    name: values.astype(np.float64) / input_scale
    for name, values in test_vectors_q.items()
}

for name, values in test_vectors_real.items():
    print(name, values.tolist())
rtl_outputs_q = {
    "Test 1 - All equal": np.array(
        [32, 32, 32, 32, 32, 32, 32, 32],
        dtype=np.int64,
    ),

    "Test 2 - Increasing": np.array(
        [1, 2, 4, 8, 16, 32, 65, 125],
        dtype=np.int64,
    ),

    "Test 3 - Decreasing": np.array(
        [125, 65, 32, 16, 8, 4, 2, 1],
        dtype=np.int64,
    ),

    "Test 4 - Negative": np.array(
        [1, 2, 4, 8, 16, 32, 64, 128],
        dtype=np.int64,
    ),

    "Test 5 - Mixed": np.array(
        [6, 0, 195, 1, 0, 48, 3, 0],
        dtype=np.int64,
    ),

    "Test 6 - Dominant": np.array(
        [242, 1, 1, 1, 1, 1, 1, 1],
        dtype=np.int64,
    ),
}

rtl_outputs_real = {
    name: values.astype(np.float64) / 256.0
    for name, values in rtl_outputs_q.items()
}

Test 1 - All equal [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Test 2 - Increasing [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 7.9375]
Test 3 - Decreasing [7.9375, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]
Test 4 - Negative [-8.0, -7.0, -6.0, -5.0, -4.0, -3.0, -2.0, -1.0]
Test 5 - Mixed [2.0, -1.0, 7.0, 0.0, -4.0, 5.0, 1.0, -2.0]
Test 6 - Dominant [7.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [ ]:
floating_comparison_rows = []

for test_name, x_real in test_vectors_real.items():
    floating_base2 = softmax_base2(x_real)
    rtl_real = rtl_outputs_real[test_name]

    absolute_error = np.abs(rtl_real - floating_base2)

    floating_comparison_rows.append({
        "Test": test_name,
        "RTL output sum": float(np.sum(rtl_real)),
        "Float Base-2 sum": float(np.sum(floating_base2)),
        "Mean absolute error": float(np.mean(absolute_error)),
        "Maximum absolute error": float(np.max(absolute_error)),
        "RTL top-1 index": int(np.argmax(rtl_real)),
        "Float top-1 index": int(np.argmax(floating_base2)),
        "Top-1 match": bool(
            np.argmax(rtl_real) == np.argmax(floating_base2)
        ),
    })

floating_comparison_df = pd.DataFrame(floating_comparison_rows)
floating_comparison_df

,Test,RTL output sum,Float Base-2 sum,Mean absolute error,Maximum absolute error,RTL top-1 index,Float top-1 index,Top-1 match
0,Test 1 - All equal,1.000000,1.0,0.000000,0.000000,0,0,True
1,Test 2 - Increasing,0.988281,1.0,0.001465,0.003219,7,7,True
2,Test 3 - Decreasing,0.988281,1.0,0.001465,0.003219,0,0,True
3,Test 4 - Negative,0.996094,1.0,0.000488,0.001961,7,7,True
4,Test 5 - Mixed,0.988281,1.0,0.001465,0.003189,2,2,True
5,Test 6 - Dominant,0.972656,1.0,0.003418,0.003501,0,0,True


In [ ]:
floating_detail_rows = []

for test_name, x_real in test_vectors_real.items():
    floating_base2 = softmax_base2(x_real)
    rtl_real = rtl_outputs_real[test_name]

    for index in range(len(x_real)):
        floating_detail_rows.append({
            "Test": test_name,
            "Index": index,
            "Input real": float(x_real[index]),
            "Float Base-2 output": float(floating_base2[index]),
            "RTL output": float(rtl_real[index]),
            "Error": float(rtl_real[index] - floating_base2[index]),
            "Absolute error": float(
                abs(rtl_real[index] - floating_base2[index])
            ),
        })

floating_detail_df = pd.DataFrame(floating_detail_rows)
floating_detail_df

,Test,Index,Input real,Float Base-2 output,RTL output,Error,Absolute error
0,Test 1 - All equal,0,1.0000,0.125000,0.125000,0.000000,0.000000
1,Test 1 - All equal,1,1.0000,0.125000,0.125000,0.000000,0.000000
2,Test 1 - All equal,2,1.0000,0.125000,0.125000,0.000000,0.000000
3,Test 1 - All equal,3,1.0000,0.125000,0.125000,0.000000,0.000000
4,Test 1 - All equal,4,1.0000,0.125000,0.125000,0.000000,0.000000
5,Test 1 - All equal,5,1.0000,0.125000,0.125000,0.000000,0.000000
6,Test 1 - All equal,6,1.0000,0.125000,0.125000,0.000000,0.000000
7,Test 1 - All equal,7,1.0000,0.125000,0.125000,0.000000,0.000000
8,Test 2 - Increasing,0,1.0000,0.004007,0.003906,-0.000101,0.000101
9,Test 2 - Increasing,1,2.0000,0.008014,0.007812,-0.000201,0.000201


In [ ]:
all_absolute_errors = floating_detail_df[
    "Absolute error"
].to_numpy()

top1_matches = floating_comparison_df[
    "Top-1 match"
].to_numpy()

print("=" * 65)
print("RTL VS FLOATING-POINT BASE-2 SOFTMAX")
print("=" * 65)
print(
    "Mean absolute error:",
    float(np.mean(all_absolute_errors)),
)
print(
    "Maximum absolute error:",
    float(np.max(all_absolute_errors)),
)
print(
    "Top-1 match rate:",
    float(np.mean(top1_matches)),
)
print(
    "Mean RTL output sum:",
    float(floating_comparison_df["RTL output sum"].mean()),
)

RTL VS FLOATING-POINT BASE-2 SOFTMAX
Mean absolute error: 0.0013834635416666633
Maximum absolute error: 0.0035011574074074077
Top-1 match rate: 1.0
Mean RTL output sum: 0.9889322916666666
